# Lesson 3: Using predefined workers

In this example we're going beyoind writing our workers.
For this we're going to extend the graph from the previous lesson.

We're going to additionally
- compile the circuit using the `tkr-pytket-worker`
- run it on a simulator using the `tkr-aer-worker` (based on the `pytket-qiskit` extension)
- calculate the expected value of our results
The workers can be installed using `uv add tkr-pytket-worker-impl tkr-aer-worker-impl`

In this example we're also covering storage and execution in a bit more detail.
First we start with the graph from before.
Since we will add more tasks we will not define the output just yet:

In [ ]:
from typing import NamedTuple

from my_example_worker import substitute, symbolic_circuit
from tierkreis.builder import Graph
from tierkreis.controller.data.models import TKR


class PytketInputs(NamedTuple):
    a: TKR[float]
    b: TKR[float]
    c: TKR[float]


graph = Graph(PytketInputs, TKR[float])

circuit = graph.task(symbolic_circuit())
substituted = graph.task(
    substitute(circuit, graph.inputs.a, graph.inputs.a, graph.inputs.a)  # type: ignore
)

Now we add the compilation step.
For generic a generic the `tkr-pytket-worker` has `compile_generic_with_fixed_pass`.
You can find a list of all available tasks [here](../worker/native_workers/pytket_worker.md)

In [ ]:
from pytket_worker import add_measure_all, compile_generic_with_fixed_pass

compiled_circuit = graph.task(
    compile_generic_with_fixed_pass(circuit, optimisation_level=graph.const(2))
)
measured = graph.task(add_measure_all(compiled_circuit))  # type: ignore

Next we will run the circuit on `aer` with the `tkr-aer-worker`s `run_circuit` (see [docs](../worker/native_workers/aer_worker.md))
And calculate the expected value.
The `expectation` tasks simply assumes the computational basis.
For other observables you could defined another task in the `my-example-worker` as before.

In [ ]:
from aer_worker import run_circuit
from pytket_worker import expectation

results = graph.task(
    run_circuit(
        circuit=compiled_circuit,  # type: ignore
        n_shots=graph.const(10),
    ),
)
exp_val = graph.task(expectation(results))
workflow = graph.finish_with_outputs(exp_val)

The Rust runtime separates workflow state from serialized assets. Both are selected in `tierkreis.toml`; callers no longer construct a storage object per run. The default configuration uses in-memory state, file-backed assets, and subprocess execution.

In [ ]:
from tierkreis.runtime import load_runtime_config, new_from_config

config = load_runtime_config()

Asset stores are named. A file-backed store can be configured as:

```toml
default_storage_name = "file"

[asset_storage.file]
type = "File"
asset_dir = "/shared/tierkreis/assets"
```

In [ ]:
config.set_log_level("info")

Executors are also named in the configuration. The subprocess executor discovers installed commands such as `tkr-pytket-worker` and `tkr-aer-worker`:

```toml
default_executor_name = "subprocess"

[executors.subprocess]
type = "Subprocess"
subprocess_storage_name = "file"
output_storage_name = "file"
```

Other configured executors can be selected through graph metadata or resource profiles.

In [ ]:
runtime = await new_from_config(config)

Save the workflow and start a run. The runtime chooses storage and executors from the configuration.

In [ ]:
with runtime:
    workflow_id = await runtime.save_workflow("quantinuum_submission", workflow)
    run_id = await runtime.start_new_run(workflow_id, {"a": -1, "b": 0, "c": 1})
    await runtime.wait_for(run_id)
    outputs = await runtime.get_outputs(run_id)
print(outputs)